# How Does Global AI Adoption Rate Compare with Productivity Change? — Phase II
### ENGE707: Data Engineering and Machine Learning Assignment — LCS-Based Prediction

Suemon Kwok (14883335), Quentin Masoe (23191960), Avathanshu Bhat (23227963)

**Dataset:** Global AI Adoption and Workforce Impact Dataset (`ai_company_adoption.csv`)

This notebook continues on from the Phase I scoping report and works through the Phase II tasks (LCS baseline, preprocessing, improved LCS system, experiments, model comparison, interpretation, and discussion).

In [1]:
import os
import zipfile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

## Task 1 - Project Phase I Summary

### 1.1 Problem restatement

- **Problem:** predict a company's reported productivity change from its AI adoption behaviour.
- **Target variable:** `productivity_change_percent` (continuous, %) — this is a **regression** problem.
- **Predictor (Phase I scope):** `ai_adoption_rate` (% of operations/processes using AI).
- **Prediction objective:** given a company-quarter's AI adoption rate (and, from Phase II onward, other available features), estimate the productivity change the company is likely to report.

In [2]:
# Load the dataset (extract from archive.zip first if the CSV isn't already alongside this notebook)
CSV_NAME = "../ai_company_adoption.csv"
ZIP_NAME = "archive.zip"

if not os.path.exists(CSV_NAME) and os.path.exists(ZIP_NAME):
    with zipfile.ZipFile(ZIP_NAME) as z:
        z.extractall(".")

df = pd.read_csv(CSV_NAME)
df.shape

(150000, 43)

### 1.2 Dataset source, size, and characteristics

In [3]:
# Quick structural recap carried over from Phase I
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Distinct companies: {df['company_id'].nunique():,}")
print(f"Countries: {df['country'].nunique()}")
print(f"Industries: {df['industry'].nunique()}")
print(f"Survey period: {df['survey_year'].min()}-{df['quarter'].min()} to {df['survey_year'].max()}-{df['quarter'].max()}")

Rows: 150,000
Columns: 43
Distinct companies: 10,000
Countries: 30
Industries: 9
Survey period: 2023-Q1 to 2026-Q4


In [4]:
# Dtype breakdown
df.dtypes.value_counts()

float64    15
int64      14
str        14
Name: count, dtype: int64

- **Source:** Global AI Adoption and Workforce Impact Dataset (Kaggle, CC0 licence).
- **Size:** 150,000 company-quarter survey responses from 10,000 distinct companies, across 30 countries and 9 industries, surveyed quarterly Q1 2023 – Q4 2026.
- **Features:** 43 columns total — a mix of numeric (AI investment, budget, maturity, satisfaction, financial metrics), categorical (industry, country, company size, tool used), and identifier columns.
- **Phase I focus variables:** `ai_adoption_rate` and `productivity_change_percent` only, per the narrowed scope agreed after Phase I lecturer feedback.

### 1.3 Main data-quality issues identified in Phase I

In [5]:
# Re-confirm the Phase I data-quality findings for the two focus variables
print("Missing values:")
print(df[["ai_adoption_rate", "productivity_change_percent"]].isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate response_id:", df["response_id"].duplicated().sum())

print("\nai_adoption_rate range:", df["ai_adoption_rate"].min(), "-", df["ai_adoption_rate"].max())
print("productivity_change_percent range:", df["productivity_change_percent"].min(), "-", df["productivity_change_percent"].max())

Missing values:
ai_adoption_rate               0
productivity_change_percent    0
dtype: int64

Duplicate rows: 0
Duplicate response_id: 0

ai_adoption_rate range: 0.0 - 100.0
productivity_change_percent range: 0.0 - 34.36


- No missing values and no duplicate records were found in either focus variable.
- Both variables sit within their valid documented ranges (0–100% and 0–34.36% respectively), so no impossible values needed correcting.
- IQR-based outlier checks flagged well under 1% of records in each variable (504 and 529 respectively) — these were treated as genuine high-adoption/high-productivity companies, not data-entry errors.
- The dataset is an **unbalanced panel**: 10,000 companies each contribute an unequal number of quarterly rows, so the 150,000 records are not fully independent observations.
- A secondary rise near 0–2% in the productivity distribution suggested a distinct "zero/low-productivity" subgroup, later linked to lower mean AI adoption (18% vs. 36% overall).

### 1.4 Why this problem is suitable for an LCS-based solution

- The target (`productivity_change_percent`) is a **continuous outcome driven by conditions across several company attributes** (adoption behaviour, workforce factors, financial context) — LCS systems are well suited to problems where the relationship is expressed as a set of localised if-then rules over feature conditions, rather than a single global function.
- The dataset is **tabular with mixed numeric and categorical features**, which matches the input format LCS implementations (e.g. eLCS, XCS) are designed for.
- Phase I already showed the AI-adoption/productivity relationship is **not purely linear or global** — the zero-productivity subgroup behaved differently from the rest of the sample. LCS's rule-based, niche-covering approach can capture this kind of subgroup structure explicitly, and its rules remain human-readable, which supports the interpretability requirement in Task 7.
- The moderate size of the dataset (150,000 rows, reducible to a manageable feature set) is within the range LCS implementations such as eLCS can realistically train on.

### Note on Task 2 scope (resolved)

Confirmed direction: keep the Phase I two-variable scope (`ai_adoption_rate` -> `productivity_change_percent`) for the **original LCS baseline** in Task 2, since the other 41 columns were already dropped during the Phase I cleaning step and never carried forward. Tasks 3/4 then widen the data to the cleaned multi-column feature space and the **improved system** narrows it again with **feature selection**; the eLCS algorithm itself is never modified.

## Task 2 - Original LCS System on Raw Dataset

### 2.1 LCS variant selection

**eLCS**, using the project's trimmed single-file implementation `elcs_custom.py` (derived from the `scikit-eLCS` / `skeLCS` eLCS code: same algorithm, with timers, tracking, reboot/pickling, debug printing and roulette selection removed). It keeps the scikit-learn `fit` / `predict` / `score` interface.

In [6]:
import itertools

# Custom, self-contained eLCS (trimmed from skeLCS) + project settings + Decision Tree factory.
# Keep elcs_custom.py in the same folder as this notebook.
from elcs_custom import (
    eLCS, make_elcs, make_decision_tree,
    N_SEEDS, TRAIN_SUBSAMPLE_SIZE, TEST_EVAL_SIZE, POPULATION_SIZE_N, LEARNING_ITERATIONS, MAX_DEPTH,
    ID_AND_TARGET_COLUMNS, OUTCOME_FAMILY_COLUMNS, NUMERIC_COLUMNS_TO_CAP, SELECTED_FEATURE_COLUMNS, STAGE_ORDER,
)

from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    multilabel_confusion_matrix,
)

### 2.2 Minimal processing required for LCS compatibility

`skeLCS.eLCS` requires:
- all feature values to be numeric (or `NaN`),
- the target (`phenotype`) to be numeric **and discrete** — this implementation always treats the phenotype as a classification target, so a continuous variable cannot be passed directly.

`ai_adoption_rate` is already numeric, so no change is needed there. `productivity_change_percent`, however, is continuous, so it is minimally processed by binning it into three quantile-based classes (Low / Medium / High) and label-encoding them as integers 0/1/2. This is the smallest possible change that makes the existing Phase I target compatible with the LCS code — no algorithmic modification is made to eLCS itself.

In [7]:
# Minimal processing: discretise the continuous target into 3 balanced classes so it is
# compatible with eLCS's classification-only phenotype requirement
df["productivity_class"] = pd.qcut(
    df["productivity_change_percent"], q=3, labels=[0, 1, 2]
).astype(int)

df["productivity_class"].value_counts().sort_index()

productivity_class
0    50023
1    50031
2    49946
Name: count, dtype: int64

In [8]:
# Baseline feature set: the single Phase I predictor, kept unmodified
X = df[["ai_adoption_rate"]].values.astype(float)
y = df["productivity_class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((120000, 1), (30000, 1))

### 2.3 LCS parameters

Default `skeLCS.eLCS` hyperparameters are used for the baseline (no tuning at this stage — tuning is reserved for the improved system in Task 4):

| Parameter | Value | Meaning |
|---|---|---|
| `learning_iterations` | 10,000 | number of training iterations (rule-learning cycles) |
| `N` | 1,000 | maximum micro-classifier population size |
| `nu` | 5 | fitness pressure exponent |
| `chi` | 0.8 | crossover probability |
| `mu` | 0.04 | mutation probability |
| `p_spec` | 0.5 | probability of specifying an attribute on covering |
| `discrete_attribute_limit` | 10 | max unique values before an attribute is treated as continuous |
| `random_state` | 42 | fixed for reproducibility |

In [9]:
# Train the original, unmodified eLCS baseline
elcs_baseline = eLCS(learning_iterations=10000, random_state=42)
elcs_baseline.fit(X_train, y_train)
print("eLCS baseline trained.")

eLCS baseline trained.


In [10]:
# Baseline performance (eLCS prediction is pure-Python and slow at scale, so train
# accuracy is estimated on a random subsample rather than all 120,000 training rows)
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_train), size=5000, replace=False)

train_acc = elcs_baseline.score(X_train[sample_idx], y_train[sample_idx])
y_pred_test = elcs_baseline.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)
coverage = elcs_baseline.get_final_instance_coverage()

print(f"Train accuracy (5,000-row sample): {train_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Final instance coverage: {coverage:.4f}")

Train accuracy (5,000-row sample): 0.5387
Test accuracy: 0.5397
Final instance coverage: 1.0000


In [11]:
print(classification_report(y_test, y_pred_test, target_names=["Low", "Medium", "High"]))

              precision    recall  f1-score   support

         Low       0.61      0.72      0.66     10005
      Medium       0.00      0.00      0.00     10006
        High       0.50      0.90      0.64      9989

    accuracy                           0.54     30000
   macro avg       0.37      0.54      0.43     30000
weighted avg       0.37      0.54      0.43     30000



/Users/avathanshubhat/Documents/AUT/Y3S2/ENGE707 - Data Engineering & AI/Assignment dataset/ENGE707_Sens-AI-tional_Project/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/avathanshubhat/Documents/AUT/Y3S2/ENGE707 - Data Engineering & AI/Assignment dataset/ENGE707_Sens-AI-tional_Project/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/avathanshubhat/Documents/AUT/Y3S2/ENGE707 - Data Engineering & AI/Assignment dataset/ENGE707_Sens-AI-tional_Project

### 2.4 Baseline results summary

Random-guess accuracy for 3 balanced classes is ~33%. The single-feature eLCS baseline should be compared against this floor once results are in — a low margin above chance would support using more of the dataset's columns, with the right ones chosen by feature selection (Task 4), as the improvement over this baseline.

## Task 3 - Data Preprocessing and Feature Engineering

In [12]:
# Target: 3-class quantile binning (Low/Medium/High). Identical for every model so comparisons are fair.
df["productivity_class"] = pd.qcut(
    df["productivity_change_percent"], q=3, labels=[0, 1, 2]
).astype(int)
print(df["productivity_class"].value_counts())

productivity_class
1    50031
0    50023
2    49946
Name: count, dtype: int64


**Preprocessing for the original eLCS (all usable columns, cleaned).** This stage keeps the broad feature space and only cleans it, so that Task 4 can isolate the effect of *feature selection* on top of it:

- **Data-quality checks:** missing values and duplicate rows are counted and reported (both are zero here, so nothing needs dropping or imputing).
- **Leakage handling:** identifiers, the column the target was cut from, and 10 "AI paid off" outcome columns (revenue growth, cost reduction, jobs created/displaced, etc.) are dropped. They measure the same kind of outcome as `productivity_change_percent`, so a model could lean on them instead of genuine adoption-side predictors.
- **Outlier handling:** the 5 core numeric predictors are winsorised (capped at their IQR fences) rather than having rows removed, so no sample size is lost.
- **Encoding:** the remaining text columns are integer-coded so eLCS can read them.

In [13]:
print("Columns with missing values:", int((df.isnull().sum() > 0).sum()))
print("Fully duplicated rows:", int(df.duplicated().sum()))

DROP_COLUMNS = ID_AND_TARGET_COLUMNS + OUTCOME_FAMILY_COLUMNS      # lists live in elcs_custom.py
PREPROCESSED_FEATURE_COLUMNS = [c for c in df.columns if c not in DROP_COLUMNS]
X_pre_df = df[PREPROCESSED_FEATURE_COLUMNS].copy()

# Winsorise: cap each core numeric predictor at its IQR fences instead of dropping rows
n_capped_total = 0
for col in NUMERIC_COLUMNS_TO_CAP:
    q1, q3 = X_pre_df[col].quantile([0.25, 0.75])
    low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    n_capped = int(((X_pre_df[col] < low) | (X_pre_df[col] > high)).sum())
    n_capped_total += n_capped
    X_pre_df[col] = X_pre_df[col].clip(lower=low, upper=high)
    print(f"  {col}: capped {n_capped} values to [{low:.2f}, {high:.2f}]")
print("Total values capped:", n_capped_total)

# Integer-code every remaining text column (no ordinal/nominal care here -- that is part of the selection step)
for col in X_pre_df.columns:
    if not pd.api.types.is_numeric_dtype(X_pre_df[col]):
        X_pre_df[col] = LabelEncoder().fit_transform(X_pre_df[col].astype(str))

PRE_FEATURE_NAMES = list(X_pre_df.columns)
X_pre = X_pre_df.values.astype(float)                  # eLCS feature matrix, original system
y_fe = df["productivity_class"].values.astype(int)     # same target for every model
print("X_pre shape:", X_pre.shape, "| y shape:", y_fe.shape)

Columns with missing values: 0
Fully duplicated rows: 0
  ai_adoption_rate: capped 504 values to [-3.23, 75.86]
  ai_maturity_score: capped 582 values to [-0.03, 0.71]
  ai_failure_rate: capped 249 values to [3.39, 48.39]
  ai_training_hours: capped 528 values to [-9.02, 63.22]
  task_automation_rate: capped 511 values to [-5.63, 45.01]
Total values capped: 2374
X_pre shape: (150000, 30) | y shape: (150000,)


No scaling was applied: eLCS matches continuous attributes by range internally, and the Decision Tree used later in the model comparison is scale-invariant, so scaling would add a processing step without changing either model's behaviour.

## Task 4 - Improved LCS-Based System

The improved system is the same, unmodified eLCS run on a **selected subset of 7 columns** taken from the cleaned data above. Feature selection is the *only* change: population size, iterations, splits and seeds match the original eLCS run.

**Why feature selection helps eLCS (reduced search space).** Every eLCS rule is an if-then condition over the available features. With ~30 columns the number of possible rule conditions is huge and most of it is noise (country codes, financial totals, calendar fields); with 7 informative columns the genetic algorithm searches a far smaller, more relevant space using the same population and iteration budget. Low-cardinality columns (`ai_adoption_stage`, 4 values; `industry`, 9 values) stay under eLCS's `discrete_attribute_limit` of 10, so they are matched by exact equality as nominal attributes instead of as a number line.

**Selected columns:** `ai_adoption_rate`, `ai_maturity_score`, `ai_failure_rate`, `ai_training_hours`, `task_automation_rate` (strongest numeric predictors, kept in their winsorised form), `ai_adoption_stage` (ordinal: none < pilot < partial < full) and `industry` (nominal, integer-coded).

### Shared evaluation protocol (Tasks 4-5)
- Seeds 0-9: a fresh 80/20 stratified split per seed, so each of the 10 trials is a genuinely different split/run (not one run repeated).
- eLCS: 1,000-row training subsample per seed, `N` = 1.5 x 1,000, `learning_iterations` = 5 x 1,000, `random_state` = seed.
- Decision Tree: trained on the full training partition per seed, `random_state` = seed.
- Every model is scored on the same 5,000-row test slice per seed, so accuracies are paired by seed.

In [14]:
# Project settings come from elcs_custom.py
CLASS_NAMES = ["Low", "Medium", "High"]
print(f"Population size N = {POPULATION_SIZE_N}, learning_iterations = {LEARNING_ITERATIONS}")

# Improved feature matrix: 7 selected columns, built from the cleaned (winsorised) frame
X_sel_df = X_pre_df[NUMERIC_COLUMNS_TO_CAP].copy()
X_sel_df["ai_adoption_stage"] = df["ai_adoption_stage"].map(STAGE_ORDER)               # ordinal, real order
X_sel_df["industry"] = LabelEncoder().fit_transform(df["industry"].astype(str))        # nominal, 9 values (< limit of 10)
assert list(X_sel_df.columns) == SELECTED_FEATURE_COLUMNS and not X_sel_df.isna().any().any()
SEL_FEATURE_NAMES = list(X_sel_df.columns)
X_sel = X_sel_df.values.astype(float)
print("Original eLCS matrix:", X_pre.shape, "| Improved eLCS matrix:", X_sel.shape)

def run_seeds(label, make_model, X, y, train_subsample=None):
    """Fit a fresh model per seed (0..N_SEEDS-1) on a fresh stratified 80/20 split; score on the fixed test slice."""
    accs, y_true_all, y_pred_all, last = [], [], [], None
    for seed in range(N_SEEDS):
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, random_state=seed, stratify=y
        )
        if train_subsample is not None:
            idx_sub = np.random.RandomState(seed).choice(len(X_tr), train_subsample, replace=False)
            X_tr, y_tr = X_tr[idx_sub], y_tr[idx_sub]
        X_ev, y_ev = X_te[:TEST_EVAL_SIZE], y_te[:TEST_EVAL_SIZE]

        model = make_model(seed)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_ev)
        acc = accuracy_score(y_ev, y_pred)
        accs.append(acc)
        y_true_all.extend(y_ev.tolist()); y_pred_all.extend(y_pred.tolist())
        last = model
        print(f"Seed {seed}: accuracy = {acc:.4f}")
    return dict(label=label, accs=accs, y_true=np.array(y_true_all), y_pred=np.array(y_pred_all), last_model=last)

def report(res):
    """Mean/std, pooled confusion matrix, classification report, per-class TP/TN/FP/FN."""
    print(f"\n{res['label']} -- mean accuracy over {N_SEEDS} seeds: "
          f"{np.mean(res['accs']):.4f} (std {np.std(res['accs']):.4f})")
    print("\nPooled confusion matrix (rows = actual, columns = predicted):")
    print(confusion_matrix(res["y_true"], res["y_pred"]))
    print("\nClassification report (pooled across all seeds):")
    print(classification_report(res["y_true"], res["y_pred"], target_names=CLASS_NAMES, zero_division=0))
    print("Per-class TP / TN / FP / FN (one-vs-rest):")
    for name, m in zip(CLASS_NAMES, multilabel_confusion_matrix(res["y_true"], res["y_pred"])):
        tn, fp, fn, tp = m.ravel()
        print(f"  {name:7s}: TP={tp:5d}  TN={tn:5d}  FP={fp:5d}  FN={fn:5d}")

def export_rules(res, filename, feature_names):
    """Export the last seed's eLCS rule population and return it as a DataFrame."""
    res["last_model"].export_final_rule_population(
        headerNames=np.array(feature_names), className="productivity_class",
        filename=filename, DCAL=True,
    )
    rules = pd.read_csv(filename)
    print(f"\nExported {len(rules)} rules to {filename}")
    print(rules.sort_values("Numerosity", ascending=False).head(5))
    return rules

def save_results(res, prefix, n_features, n_rules=None):
    """Write <prefix>_seed_results.csv (seed, model, accuracy) + <prefix>_results.csv summary."""
    pd.DataFrame({"seed": range(N_SEEDS), "model": res["label"], "accuracy": res["accs"]}
                 ).to_csv(f"{prefix}_seed_results.csv", index=False)
    pd.DataFrame([{"model": res["label"], "mean_accuracy": np.mean(res["accs"]),
                   "std_accuracy": np.std(res["accs"]), "n_seeds": N_SEEDS,
                   "n_features": n_features, "n_rules_last_seed": n_rules}]
                 ).to_csv(f"{prefix}_results.csv", index=False)
    print(f"Saved {prefix}_seed_results.csv and {prefix}_results.csv")

Population size N = 1500, learning_iterations = 5000


Original eLCS matrix: (150000, 30) | Improved eLCS matrix: (150000, 7)


### Improved eLCS (feature selection, 7 columns): 10-seed run

In [15]:
res_improved = run_seeds(
    "Improved eLCS (feature selection, 7 features)",
    make_elcs,                       # same eLCS settings as the original; the change is X_sel (7 columns)
    X_sel, y_fe, train_subsample=TRAIN_SUBSAMPLE_SIZE,
)
report(res_improved)

Seed 0: accuracy = 0.6124


KeyboardInterrupt: 

In [ ]:
rules_improved = export_rules(res_improved, "task4_elcs_feature_selection_avathanshu_rules.csv", SEL_FEATURE_NAMES)
save_results(res_improved, "task4_elcs_feature_selection_avathanshu", X_sel.shape[1], len(rules_improved))


Exported 1121 rules to task4_elcs_feature_selection_avathanshu_rules.csv
                                      Specified Values  \
183  [0.09837500000000002,0.283625], [-1.5062500000...   
291  [7.01585,42.999500000000005], [23.833875,29.54...   
103         [0.047125000000000056,0.48887499999999995]   
93   [26.109924999999997,41.850075], [-1.5062500000...   
75   [26.109924999999997,41.850075], [-10.985300000...   

                             Specified Attribute Names  productivity_class  \
183  ai_maturity_score, task_automation_rate, ai_ad...                 0.0   
291  ai_adoption_rate, ai_failure_rate, task_automa...                 1.0   
103                                  ai_maturity_score                 1.0   
93               ai_failure_rate, task_automation_rate                 0.0   
75                  ai_failure_rate, ai_training_hours                 0.0   

      Fitness  Accuracy  Numerosity  Avg Match Set Size  TimeStamp GA  \
183  0.255551  0.761194           6

## Task 5 - Experiments

### 5.1 Validation strategy

Repeated hold-out over **10 seeds (0-9)**: each seed uses its own 80/20 stratified split (and its own model seed), giving 10 independent trials per model rather than one run repeated. The same split and the same 5,000-row test slice are used for every model within a seed, so accuracies are paired by seed for the significance tests in 5.4.

### 5.2 Run the remaining models (10 seeds each)

In [ ]:
# Original, unmodified eLCS on the cleaned full feature set (same protocol as the improved run)
res_original = run_seeds(
    "Original eLCS (preprocessed, all cleaned features)",
    make_elcs,
    X_pre, y_fe, train_subsample=TRAIN_SUBSAMPLE_SIZE,
)
report(res_original)
rules_original = export_rules(res_original, "task3_elcs_preprocessed_avathanshu_rules.csv", PRE_FEATURE_NAMES)
save_results(res_original, "task3_elcs_preprocessed_avathanshu", X_pre.shape[1], len(rules_original))

Seed 0: accuracy = 0.5938
Seed 1: accuracy = 0.4932
Seed 2: accuracy = 0.3462
Seed 3: accuracy = 0.4686
Seed 4: accuracy = 0.4892
Seed 5: accuracy = 0.5352
Seed 6: accuracy = 0.3416
Seed 7: accuracy = 0.4766
Seed 8: accuracy = 0.5018
Seed 9: accuracy = 0.3878

Original eLCS (preprocessed, all cleaned features) -- mean accuracy over 10 seeds: 0.4634 (std 0.0772)

Pooled confusion matrix (rows = actual, columns = predicted):
[[ 8047  2213  6298]
 [ 4897  3097  8835]
 [ 2007  2580 12026]]

Classification report (pooled across all seeds):
              precision    recall  f1-score   support

         Low       0.54      0.49      0.51     16558
      Medium       0.39      0.18      0.25     16829
        High       0.44      0.72      0.55     16613

    accuracy                           0.46     50000
   macro avg       0.46      0.46      0.44     50000
weighted avg       0.46      0.46      0.44     50000

Per-class TP / TN / FP / FN (one-vs-rest):
  Low    : TP= 8047  TN=26538  FP= 

In [ ]:
# Decision Tree: same 7 source columns, integer-coded categoricals (trees only need splittable values),
# trained on the FULL training partition per seed with random_state = seed.
X_dt_df = df[SELECTED_FEATURE_COLUMNS].copy()
for col in ["ai_adoption_stage", "industry"]:
    X_dt_df[col] = LabelEncoder().fit_transform(X_dt_df[col].astype(str))
X_dt = X_dt_df.values.astype(float)

res_dt = run_seeds(
    "Decision Tree (7 selected features)",
    make_decision_tree,
    X_dt, y_fe, train_subsample=None,
)
report(res_dt)

importances = pd.DataFrame({
    "feature": SELECTED_FEATURE_COLUMNS,
    "importance": res_dt["last_model"].feature_importances_,
}).sort_values("importance", ascending=False)
print("\nFeature importances (final seed's tree):")
print(importances.to_string(index=False))
importances.to_csv("task6_decision_tree_feature_importances.csv", index=False)
save_results(res_dt, "task6_decision_tree", X_dt.shape[1])

Seed 0: accuracy = 0.6356
Seed 1: accuracy = 0.6142
Seed 2: accuracy = 0.6246
Seed 3: accuracy = 0.6144
Seed 4: accuracy = 0.6220
Seed 5: accuracy = 0.6238
Seed 6: accuracy = 0.6204
Seed 7: accuracy = 0.6398
Seed 8: accuracy = 0.6238
Seed 9: accuracy = 0.6088

Decision Tree (7 selected features) -- mean accuracy over 10 seeds: 0.6227 (std 0.0090)

Pooled confusion matrix (rows = actual, columns = predicted):
[[11502  4381   675]
 [ 4407  8226  4196]
 [  658  4546 11409]]

Classification report (pooled across all seeds):
              precision    recall  f1-score   support

         Low       0.69      0.69      0.69     16558
      Medium       0.48      0.49      0.48     16829
        High       0.70      0.69      0.69     16613

    accuracy                           0.62     50000
   macro avg       0.62      0.62      0.62     50000
weighted avg       0.62      0.62      0.62     50000

Per-class TP / TN / FP / FN (one-vs-rest):
  Low    : TP=11502  TN=28377  FP= 5065  FN= 5056


### 5.3 Evaluation summary

In [ ]:
all_results = [res_original, res_improved, res_dt]
wide = pd.DataFrame({r["label"]: r["accs"] for r in all_results}, index=pd.Index(range(N_SEEDS), name="seed"))
summary = wide.agg(["mean", "std"]).T.sort_values("mean", ascending=False)
summary.to_csv("task5_summary.csv")
summary.round(4)

,mean,std
Decision Tree (7 selected features),0.6227,0.0095
"Improved eLCS (feature selection, 7 features)",0.5884,0.0124
"Original eLCS (preprocessed, all cleaned features)",0.4634,0.0814


### 5.4 Statistical significance testing

Every model is scored on the same test rows for each seed, so accuracies are **paired by seed**. Tests used: paired t-test (assumes roughly normal differences) and the Wilcoxon signed-rank test (non-parametric; with n=10 the smallest possible two-sided p is ~0.002). Holm step-down correction is applied across all pairwise comparisons.

In [ ]:
def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1, run)
    return adj

rows = []
for a, b in itertools.combinations(wide.columns, 2):
    d = wide[a] - wide[b]
    t_p = stats.ttest_rel(wide[a], wide[b]).pvalue
    try:
        w_p = stats.wilcoxon(wide[a], wide[b]).pvalue
    except ValueError:
        w_p = np.nan
    rows.append(dict(model_a=a, model_b=b, mean_diff=d.mean(), ttest_p=t_p, wilcoxon_p=w_p))
pairwise = pd.DataFrame(rows)
pairwise["ttest_p_holm"] = holm(pairwise["ttest_p"])
pairwise["wilcoxon_p_holm"] = holm(pairwise["wilcoxon_p"].fillna(1))
pairwise["significant_holm_0.05"] = pairwise["ttest_p_holm"] < 0.05
pairwise.to_csv("task5_pairwise_tests.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
print(pairwise.round(5).to_string(index=False))

                                           model_a                                       model_b  mean_diff  ttest_p  wilcoxon_p  ttest_p_holm  wilcoxon_p_holm  significant_holm_0.05
Original eLCS (preprocessed, all cleaned features) Improved eLCS (feature selection, 7 features)   -0.12496  0.00080     0.00195       0.00080          0.00586                   True
Original eLCS (preprocessed, all cleaned features)           Decision Tree (7 selected features)   -0.15934  0.00012     0.00195       0.00023          0.00586                   True
     Improved eLCS (feature selection, 7 features)           Decision Tree (7 selected features)   -0.03438  0.00000     0.00195       0.00001          0.00586                   True
